# Track 1 automatic segmentation batch
Automatic Grounding DINO + SAM2 prompts from released Track 1 videos. T4 FP16
uses consistent stored-memory precision; retry FP32 on numerical failure.
Retain per-frame finite checks and exact original-frame coverage. No manual labels.


In [ ]:
import os,sys,subprocess,json,urllib.request,hashlib
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','transformers==5.3.0','h5py','opencv-python<4.12'],check=True)
env=os.environ.copy();env['SAM2_BUILD_CUDA']='0'
subprocess.run(['git','clone','--depth','1','https://github.com/facebookresearch/sam2.git','/tmp/sam2_source'],check=True)
sam_revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd='/tmp/sam2_source',text=True).strip()
predictor_source=Path('/tmp/sam2_source/sam2/sam2_video_predictor.py')
original=predictor_source.read_text()
predictor_backup=original
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','-e','/tmp/sam2_source'],env=env,check=True)
checkpoint=Path('/tmp/sam2.1_hiera_large.pt')
urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt',checkpoint)


In [ ]:
Path('/tmp/segment_video.py').write_text('"""Automatic Track 1 video masks from released text prompts; no manual labels.\n\nGrounding DINO chooses a shared human/object seed frame. SAM2 propagates in both\ndirections. Output H5 follows the official wild-video mask schema exactly.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport cv2\nimport h5py\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom transformers import AutoProcessor, AutoModelForZeroShotObjectDetection\nfrom sam2.build_sam import build_sam2_video_predictor\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--video\',type=Path,required=True)\n    p.add_argument(\'--object-prompt\',required=True)\n    p.add_argument(\'--sam2-checkpoint\',required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--expected-frames\',type=int,required=True)\n    p.add_argument(\'--precision\',choices=[\'float32\',\'float16\'],default=\'float32\')\n    p.add_argument(\'--object-class\')\n    p.add_argument(\'--semantic\',action=\'store_true\')\n    a = p.parse_args()\n    suffix=\'.0.color.mp4\'\n    if not a.video.name.endswith(suffix):\n        p.error(\'Name the input <sequence>.0.color.mp4\')\n    sequence=a.video.name[:-len(suffix)]\n    a.output.mkdir(parents=True,exist_ok=True)\n    frames=a.output/\'frames\';frames.mkdir(exist_ok=True)\n    cap=cv2.VideoCapture(str(a.video));fps=float(cap.get(cv2.CAP_PROP_FPS))\n    index=0\n    while True:\n        ok,image=cap.read()\n        if not ok:break\n        if not cv2.imwrite(str(frames/f\'{index:06d}.jpg\'),image,[cv2.IMWRITE_JPEG_QUALITY,95]):\n            raise OSError(\'Frame write failed\')\n        height,width=image.shape[:2];index+=1\n    cap.release()\n    if index!=a.expected_frames:\n        raise ValueError(f\'Decoded {index} frames, expected {a.expected_frames}\')\n    if a.semantic:\n        from select_seed import select_seed\n        seed,candidates,detector_id=select_seed(frames,index,height,width,a.object_prompt,a.object_class)\n    else:\n        detector_id=\'IDEA-Research/grounding-dino-tiny\'\n        processor=AutoProcessor.from_pretrained(detector_id)\n        detector=AutoModelForZeroShotObjectDetection.from_pretrained(detector_id).to(\'cuda\').eval()\n        candidates=[]\n        for frame in sorted({min(index-1,t) for t in (0,15,30,60,120)}):\n            image=Image.open(frames/f\'{frame:06d}.jpg\').convert(\'RGB\')\n            pair=[]\n            for prompt in (\'a person.\',a.object_prompt):\n                inputs=processor(images=image,text=prompt,return_tensors=\'pt\').to(\'cuda\')\n                with torch.inference_mode():prediction=detector(**inputs)\n                result=processor.post_process_grounded_object_detection(prediction,inputs.input_ids,\n                    threshold=0.15,text_threshold=0.15,target_sizes=[(height,width)])[0]\n                keep=[i for i,b in enumerate(result[\'boxes\']) if (b[2]-b[0])>=8 and (b[3]-b[1])>=8]\n                if not keep:break\n                best=max(keep,key=lambda i:float(result[\'scores\'][i]))\n                pair.append({\'box\':result[\'boxes\'][best].detach().cpu().tolist(),\n                             \'confidence\':float(result[\'scores\'][best])})\n            if len(pair)==2:\n                candidates.append({\'frame\':frame,\'human\':pair[0],\'object\':pair[1],\n                                   \'score\':pair[0][\'confidence\']+pair[1][\'confidence\']})\n        del detector,processor\n        torch.cuda.empty_cache()\n        if not candidates:\n            raise ValueError(\'No automatic human/object seed pair found; do not fabricate masks\')\n        seed=max(candidates,key=lambda c:c[\'score\'])\n    predictor=build_sam2_video_predictor(\'configs/sam2.1/sam2.1_hiera_l.yaml\',\n        a.sam2_checkpoint,device=\'cuda\',apply_postprocessing=False)\n    seen=set();areas={};preview_ids=set(np.linspace(0,index-1,8).astype(int).tolist())\n    with torch.inference_mode(), torch.autocast(\'cuda\',dtype=torch.float16,enabled=a.precision==\'float16\'):\n        state=predictor.init_state(str(frames),offload_video_to_cpu=True,offload_state_to_cpu=True)\n        for obj_id,role in ((0,\'human\'),(1,\'object\')):\n            predictor.add_new_points_or_box(state,frame_idx=seed[\'frame\'],obj_id=obj_id,\n                box=np.asarray(seed[role][\'box\'],dtype=np.float32))\n        with h5py.File(a.output/\'masks.h5\',\'w\') as h:\n            h.attrs.update(schema=\'v2d.cari4d.wild_masks.v1\',sequence=sequence,\n                           frame_count=index,width=width,height=height)\n            group=h.create_group(sequence)\n            for reverse in (False,True):\n                for frame,obj_ids,logits in predictor.propagate_in_video(state,\n                        start_frame_idx=seed[\'frame\'],reverse=reverse):\n                    if frame in seen:continue\n                    if not torch.isfinite(logits).all():\n                        raise ValueError(f\'Non-finite SAM2 output at frame {frame}\')\n                    masks={int(obj):logits[i,0].detach().cpu().numpy()>0 for i,obj in enumerate(obj_ids)}\n                    if set(masks)!={0,1}:raise ValueError(\'Incomplete tracked object set\')\n                    for obj,role in ((0,\'person_mask\'),(1,\'obj_rend_mask\')):\n                        group.create_dataset(f\'{frame:06d}-k0.{role}.png\',\n                            data=masks[obj].astype(np.uint8)*255,compression=\'lzf\',shuffle=True)\n                    areas[frame]=[int(masks[0].sum()),int(masks[1].sum())]\n                    if frame in preview_ids:\n                        rgb=np.asarray(Image.open(frames/f\'{frame:06d}.jpg\').convert(\'RGB\')).copy()\n                        rgb[masks[0]]=(rgb[masks[0]]*.6+np.array([40,130,255])*.4).astype(np.uint8)\n                        rgb[masks[1]]=(rgb[masks[1]]*.5+np.array([255,70,40])*.5).astype(np.uint8)\n                        Image.fromarray(rgb).save(a.output/f\'preview_{frame:06d}.jpg\')\n                    if frame==seed[\'frame\']:\n                        rgba=np.asarray(Image.open(frames/f\'{frame:06d}.jpg\').convert(\'RGBA\')).copy()\n                        rgba[:,:,3]=masks[1].astype(np.uint8)*255\n                        ys,xs=np.where(masks[1])\n                        if len(xs)<32:raise ValueError(\'Object seed mask is empty or degenerate\')\n                        margin=int(.15*max(xs.max()-xs.min(),ys.max()-ys.min()))+1\n                        box=(max(0,xs.min()-margin),max(0,ys.min()-margin),\n                             min(width,xs.max()+margin+1),min(height,ys.max()+margin+1))\n                        Image.fromarray(rgba).crop(box).save(a.output/\'object_observation.png\')\n                    seen.add(frame)\n                    if len(seen)%50==0:print(f\'Masks: {len(seen)}/{index}\',flush=True)\n    if seen!=set(range(index)):raise ValueError(\'Masks do not cover every original frame\')\n    report={\'sequence\':sequence,\'frames\':index,\'fps\':fps,\'width\':width,\'height\':height,\n        \'seed\':seed,\'seed_candidates\':candidates,\'detector\':detector_id,\n        \'object_prompt\':a.object_prompt,\'precision\':a.precision,\'semantic_selection\':a.semantic,\'object_class\':a.object_class,\'manual_annotations_used\':False,\n        \'frame_mask_areas\':{str(k):v for k,v in sorted(areas.items())},\n        \'video_sha256\':hashlib.sha256(a.video.read_bytes()).hexdigest()}\n    (a.output/\'segmentation_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(\'Automatic segmentation complete:\',a.output)\n\n\nif __name__==\'__main__\':main()\n')
Path('/tmp/select_seed.py').write_text('"""Select automatic video seeds using grounded proposals and semantic margins."""\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom transformers import AutoProcessor,AutoModel,AutoModelForZeroShotObjectDetection\n\nDESCRIPTIONS={\n \'hula_hoop\':[\'a thin circular hula hoop\',\'a plastic exercise hoop\'],\n \'big_red_bowl\':[\'a red plastic bowl\',\'a large red mixing bowl\'],\n \'iron\':[\'a blue electric clothes iron\',\'a metallic steam iron with blue accents\'],\n \'white_desk\':[\'a white rectangular office desk with two legs\',\'a white standing desk\'],\n \'black_pan\':[\'a black frying pan with a handle\',\'a black cooking pan\'],\n \'foam_grass_block\':[\'a green and brown pixelated Minecraft toy cube\',\'a foam cube with green and brown squares\'],\n \'paint_roller\':[\'a white paint roller with a blue handle\',\'a paint roller tool with a handle\'],\n \'pink_foam_roll\':[\'a pink cylindrical foam exercise roller\',\'a pink foam cylinder with a hole\'],\n \'short_wood_stool\':[\'a short wooden four-legged stool\',\'a wooden square stool\'],\n \'white_laptop_cart\':[\'a white rolling laptop cart with a central column and wheels\',\'a white mobile laptop stand with casters\'],\n}\nQUERIES={\n \'iron\':[\'a clothes iron.\',\'a blue electric iron.\'],\n \'foam_grass_block\':[\'a green brown cube.\',\'a Minecraft block.\',\'a foam cube.\'],\n \'white_desk\':[\'a white desk.\',\'an office desk.\'],\n \'white_laptop_cart\':[\'a rolling laptop stand.\',\'a white cart with wheels.\'],\n \'paint_roller\':[\'a paint roller.\',\'a paint roller with handle.\'],\n}\nDISTRACTORS=[\'a camera tripod\',\'an office chair\',\'a curtain\',\'a person\',\'a slipper\',\n             \'a long metal rail\',\'a wall\',\'a wooden floor\',\'a table lamp\',\'an ironing board\']\n\n\ndef select_seed(frames:Path,frame_count:int,height:int,width:int,prompt:str,object_class:str):\n    if object_class not in DESCRIPTIONS:raise ValueError(\'Unknown released object class\')\n    detector_id=\'IDEA-Research/grounding-dino-tiny\'\n    processor=AutoProcessor.from_pretrained(detector_id)\n    detector=AutoModelForZeroShotObjectDetection.from_pretrained(detector_id).to(\'cuda\').eval()\n    candidates=[];crops=[]\n    queries=list(dict.fromkeys(QUERIES.get(object_class,[DESCRIPTIONS[object_class][0]+\'.\'])+[prompt]))\n    sampled=sorted(set(np.linspace(0,frame_count-1,12).astype(int).tolist()+[min(frame_count-1,x) for x in (0,15,30,60,120)]))\n    for frame in sampled:\n        image=Image.open(frames/f\'{frame:06d}.jpg\').convert(\'RGB\')\n        def detect(text):\n            inputs=processor(images=image,text=text,return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode():output=detector(**inputs)\n            return processor.post_process_grounded_object_detection(output,inputs.input_ids,\n                threshold=.12,text_threshold=.12,target_sizes=[(height,width)])[0]\n        people=detect(\'a person.\')\n        if not len(people[\'boxes\']):continue\n        human_id=int(people[\'scores\'].argmax())\n        human={\'box\':people[\'boxes\'][human_id].cpu().tolist(),\'confidence\':float(people[\'scores\'][human_id])}\n        seen=[]\n        for query in queries:\n            result=detect(query)\n            for box,confidence in zip(result[\'boxes\'].cpu().numpy(),result[\'scores\'].cpu().numpy()):\n                x0,y0,x1,y1=box\n                if x1-x0<8 or y1-y0<8:continue\n                if any(np.max(np.abs(box-prev))<3 for prev in seen):continue\n                seen.append(box)\n                margin=.06*max(x1-x0,y1-y0)\n                crop=image.crop((max(0,int(x0-margin)),max(0,int(y0-margin)),min(width,int(x1+margin+1)),min(height,int(y1+margin+1))))\n                candidates.append({\'frame\':frame,\'human\':human,\'object\':{\'box\':box.astype(float).tolist(),\'confidence\':float(confidence)},\'query\':query})\n                crops.append(crop)\n    del processor,detector;torch.cuda.empty_cache()\n    if not candidates:raise ValueError(\'No automatic grounded candidates\')\n    model_id=\'google/siglip2-base-patch16-224\'\n    processor=AutoProcessor.from_pretrained(model_id)\n    model=AutoModel.from_pretrained(model_id).to(\'cuda\').eval()\n    target_text=DESCRIPTIONS[object_class]\n    other_text=[text for key,values in DESCRIPTIONS.items() if key!=object_class for text in values]+DISTRACTORS\n    texts=[\'This is a photo of \'+text+\'.\' for text in target_text+other_text]\n    for start in range(0,len(crops),16):\n        inputs=processor(text=texts,images=crops[start:start+16],padding=\'max_length\',max_length=64,truncation=True,return_tensors=\'pt\').to(\'cuda\')\n        with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.float16):logits=model(**inputs).logits_per_image.float().cpu().numpy()\n        if not np.isfinite(logits).all():raise ValueError(\'Nonfinite semantic ranking logits\')\n        for offset,row in enumerate(logits):\n            item=candidates[start+offset]\n            positive=float(row[:len(target_text)].max());negative=float(row[len(target_text):].max())\n            item.update(semantic_margin=positive-negative,semantic_target_logit=positive,\n                        semantic_best_label=(target_text+other_text)[int(row.argmax())],semantic_model=model_id)\n            item[\'score\']=positive-negative+.1*positive+.25*item[\'object\'][\'confidence\']\n    eligible=[c for c in candidates if c[\'semantic_margin\']>0]\n    if not eligible:raise ValueError(\'No candidate beats the released alternative classes/distractors\')\n    seed=max(eligible,key=lambda c:c[\'score\'])\n    del model,processor;torch.cuda.empty_cache()\n    return seed,candidates,detector_id\n')
semantic=True
episode_ids=[6, 8, 9, 16, 17, 19, 27, 28]
base='https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_1/'
with urllib.request.urlopen(base+'meta/episodes_metadata.jsonl') as response: metadata={x['episode_index']:x for x in map(json.loads,response.read().decode().splitlines())}
with urllib.request.urlopen(base+'meta/episodes.jsonl') as response: episodes={x['episode_index']:x for x in map(json.loads,response.read().decode().splitlines())}
import shutil,time
results=[]
for episode_id in episode_ids:
 m=metadata[episode_id];episode=episodes[episode_id]
 video=Path('/tmp')/(m['sequence_id']+'.0.color.mp4')
 urllib.request.urlretrieve(base+f'videos/chunk-000/observation.images.exo_camera/episode_{episode_id:06d}.mp4',video)
 output=Path('/kaggle/working')/f'episode_{episode_id:06d}'
 started=time.monotonic();success=False
 for precision in ('float16','float32'):
  predictor_source.write_text(predictor_backup.replace('torch.bfloat16','torch.'+precision))
  # Avoid stale bytecode when switching dtypes in a fresh child interpreter.
  for cache in predictor_source.parent.glob('__pycache__/sam2_video_predictor.*.pyc'):cache.unlink()
  command=[sys.executable,'/tmp/segment_video.py','--video',str(video),'--object-prompt',m['object_prompt'],
   '--sam2-checkpoint',str(checkpoint),'--output',str(output),'--expected-frames',str(episode['length']),'--precision',precision]
  if semantic:command+=['--semantic','--object-class',m['object']]
  process=subprocess.run(command,env=env)
  if process.returncode==0:success=True;break
  print('Retrying failed segmentation in full precision',episode_id,flush=True)
 result={'episode':episode_id,'success':success,'precision':precision,'elapsed_seconds':time.monotonic()-started}
 results.append(result)
 Path('/kaggle/working/batch_report.json').write_text(json.dumps(results,indent=2))
 if success:
  shutil.rmtree(output/'frames');video.unlink()
 print('EPISODE_RESULT',result,flush=True)
Path('/kaggle/working/model_manifest.json').write_text(json.dumps({'sam2_revision':sam_revision,
 'sam2_weight_sha256':hashlib.sha256(checkpoint.read_bytes()).hexdigest(),'track':'track_1',
 'precision_policy':'FP16 with matching stored memory; retry FP32 on failure'},indent=2))
if any(not x['success'] for x in results):raise RuntimeError('Some episodes failed; inspect batch_report.json')
